In [ ]:
# ============================================
# Parkinson's Disease vs Control EEG Classifier
# Full Pipeline: BIDS loading, Preprocessing, ICA, Feature Extraction, ML
# Dataset: ds002778 (BIDS format)
# ============================================

import os
import os.path as op
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mne
from mne.datasets import sample
from mne_bids import BIDSPath, read_raw_bids, get_entity_vals
from mne.preprocessing import ICA, create_eog_epochs
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
import os
import openneuro

# Define BIDS root folder
bids_root = "PD3"
os.makedirs(bids_root, exist_ok=True)

# Download dataset (all subjects, all sessions)
#openneuro.download(dataset="ds003506", target_dir=bids_root)

print("✅ Download complete.")


In [ ]:
import os

subjects = [d for d in os.listdir(bids_root) if d.startswith("sub-")]
print(f"👥 Subjects found ({len(subjects)} total):")
for subject in subjects:
    print(f" - {subject}")


In [ ]:
import os
import pandas as pd
import torch
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Load TSV participants file
tsv_path = "PD3/56participants.tsv"
tsv_df = pd.read_csv(tsv_path, sep="\t")
tsv_df.columns = tsv_df.columns.str.strip()

# Format participant IDs (e.g., sub-001)
def format_participant_id(x):
    try:
        return f"sub-{int(str(x).strip().replace('sub-', '')):03d}"
    except:
        return np.nan

tsv_df["participant_id"] = tsv_df["participant_id"].apply(format_participant_id)

# Encode sex (Male=1, Female=0)
le_gender = LabelEncoder()
tsv_df["sex_encoded"] = le_gender.fit_transform(tsv_df["sex"].astype(str).str.strip())

# Initialize data holders
features = []
labels = []
subject_ids = []

bids_root = "PD3"  # EEG folder

for subject in sorted(os.listdir(bids_root)):
    subj_path = os.path.join(bids_root, subject)
    if not os.path.isdir(subj_path) or not subject.startswith("sub-"):
        continue

    subj_row = tsv_df[tsv_df["participant_id"] == subject]
    if subj_row.empty:
        print(f"❌ Skipping {subject}: not in TSV metadata")
        continue

    subj_row = subj_row.iloc[0]
    age = subj_row.get("age", np.nan)
    gender = subj_row.get("sex_encoded", np.nan)

    # Loop sessions in folder
    for session in os.listdir(subj_path):
        if not session.startswith("ses-"):
            continue
        session_path = os.path.join(subj_path, session)
        if not os.path.isdir(session_path):
            continue

        # Determine label depending on group and medication status for this session
        group = subj_row["Group"].strip()  # 'CTL' or 'PD'
        label = None

        if group == "CTL":
            label = 0  # Control for any session
        elif group == "PD":
            # Medication status column depends on session
            if session == "ses-01":
                med_status = subj_row.get("sess1_Med", "").strip().upper()
            elif session == "ses-02":
                med_status = subj_row.get("sess2_Med", "").strip().upper()
            else:
                med_status = ""

            if med_status == "ON":
                label = 1  # PD-On
            elif med_status == "OFF":
                label = 2  # PD-Off

        if label is None:
            print(f"⚠️ Skipping {subject} {session}: cannot determine label")
            continue

        feature_vector = [age, gender]
        features.append(np.array(feature_vector))
        labels.append(label)
        subject_ids.append(f"{subject}_{session}")

# Convert to tensors
features_tensor = torch.tensor(features, dtype=torch.float32)
labels_tensor = torch.tensor(labels, dtype=torch.long)

print(f"✅ Features shape: {features_tensor.shape}")
print(f"✅ Labels shape: {labels_tensor.shape}")
print(f"🧮 Label counts: {np.bincount(labels_tensor.numpy())}")


## Step 1: Define Feature Extraction Function

In [ ]:
import numpy as np
from scipy.stats import skew
from sklearn.decomposition import PCA

def extract_band_power_epochs(
    epochs,
    bands=[(1, 4), (4, 8), (8, 13), (13, 30)],  # Frequency bands
    compress=False,  # If True, aggregate band power stats per epoch
    apply_pca=True,  # Apply PCA to reduce dimensions
    pca_components=50  # Number of PCA components to retain
):
    """
    Extract band power features from EEG epochs.

    Parameters:
    - epochs: mne.Epochs object
    - bands: list of (fmin, fmax) frequency tuples
    - compress: If True, use mean, std, skew aggregated over channels per band (lower-dim features).
                If False, return band power for each channel and each band (higher-dim).
    - apply_pca: If True, apply PCA to reduce dimensionality.
    - pca_components: Number of PCA components to keep (int or None for auto).

    Returns:
    - X: ndarray of shape (n_epochs, n_features) feature matrix.
    - feature_names: list of feature names (empty if PCA applied).
    - pca_model: trained PCA object or None.
    """
    # Compute PSD using Welch method (recommended)
    psd = epochs.compute_psd(method='welch')
    psd_data, freqs = psd.get_data(return_freqs=True)  # (n_epochs, n_channels, n_freqs)
    ch_names = epochs.info['ch_names']

    X = []

    if compress:
        feature_names = []
        for fmin, fmax in bands:
            band_label = f"{int(fmin)}-{int(fmax)}Hz"
            feature_names.extend([f"{band_label}_mean", f"{band_label}_std", f"{band_label}_skew"])

        for epoch_psd in psd_data:
            features = []
            for fmin, fmax in bands:
                freq_idx = np.logical_and(freqs >= fmin, freqs <= fmax)
                band_power = epoch_psd[:, freq_idx].mean(axis=1)  # mean over frequencies per channel
                # Aggregate statistics across channels
                features.extend([
                    np.mean(band_power),      # mean across channels
                    np.std(band_power),       # std dev across channels
                    skew(band_power)          # skewness across channels
                ])
            X.append(features)

    else:
        feature_names = []
        for fmin, fmax in bands:
            for ch in ch_names:
                feature_names.append(f"{int(fmin)}-{int(fmax)}Hz_{ch}")

        for epoch_psd in psd_data:
            features = []
            for fmin, fmax in bands:
                freq_idx = np.logical_and(freqs >= fmin, freqs <= fmax)
                band_power = epoch_psd[:, freq_idx].mean(axis=1)  # mean over frequencies per channel
                features.extend(band_power.tolist())  # raw band power per channel
            X.append(features)

    X = np.array(X)

    pca_model = None
    if apply_pca:
        n_samples, n_features = X.shape
        if pca_components is None or pca_components > min(n_samples, n_features):
            pca_components = min(n_samples, n_features)
            print(f"Auto-adjusting PCA components to {pca_components}")

        print(f"Applying PCA: reducing {X.shape[1]} → {pca_components} features")
        pca_model = PCA(n_components=pca_components, random_state=42)
        X = pca_model.fit_transform(X)
        feature_names = [f"PCA_{i+1}" for i in range(X.shape[1])]

    return X, feature_names, pca_model


## Step 2: Preprocess and Extract for Subject

In [ ]:
import time
import psutil
import numpy as np
import mne
import os
from mne.preprocessing import ICA, create_eog_epochs

# from your_module import extract_band_power_epochs  # <-- Ensure this is imported

def categorize_session(session_name):
    session_name = session_name.lower()
    if 'on' in session_name:
        return 'pd-on'
    elif 'off' in session_name:
        return 'pd-off'
    elif 'hc' in session_name or 'ctl' in session_name:
        return 'ctl'
    else:
        return 'unknown'

def preprocess_and_extract_features(raw_data, duration=2.0, overlap=1.0):
    print("Starting preprocessing and feature extraction...")

    print("Filtering data (1-40 Hz bandpass and 60Hz notch)...")
    raw_data.filter(l_freq=1, h_freq=50, fir_design='firwin', verbose=False)
    raw_data.notch_filter(freqs=60, verbose=False)
    raw_data.set_eeg_reference('average', verbose=False)

    if raw_data.info['bads']:
        print(f"Interpolating bad channels: {raw_data.info['bads']}")
        raw_data.interpolate_bads()

    print("Running ICA...")
    ica = ICA(n_components=20, random_state=97, max_iter=800, verbose=False)
    ica.fit(raw_data)

    eog_channels = [ch for ch in raw_data.info['ch_names'] if ch.startswith('EXG')]
    if eog_channels:
        print(f"Detecting EOG artifacts in channels: {eog_channels}")
        eog_epochs = create_eog_epochs(raw_data, ch_name=eog_channels[0])
        eog_inds, _ = ica.find_bads_eog(eog_epochs, ch_name=eog_channels[0])
        ica.exclude = eog_inds
        print(f"Excluding ICA components: {ica.exclude}")
        raw_data = ica.apply(raw_data)
    else:
        print("No EOG channels found; skipping EOG artifact removal.")

    print(f"Epoching data: duration={duration}s, overlap={overlap}s")
    epochs = mne.make_fixed_length_epochs(raw_data, duration=duration, overlap=overlap, preload=True, reject_by_annotation=True)

    if len(epochs) == 0:
        print("⚠️ No epochs created, returning empty results.")
        return None, None

    print("Extracting band power features...")
    X, feature_names, _ = extract_band_power_epochs(epochs, apply_pca=False)
    return X, feature_names

def process_subject_session(subject, session, label, bids_root, duration=2.0, overlap=1.0):
    start_time = time.time()
    try:
        print(f"\n🚀 Processing subject={subject}, session={session}")
        subject_dir = os.path.join(bids_root, subject)
        session_dir = os.path.join(subject_dir, session, "eeg")
        set_files = [f for f in os.listdir(session_dir) if f.endswith('.set')]

        if not set_files:
            raise FileNotFoundError(f"No .set files found in {session_dir}")

        set_path = os.path.join(session_dir, set_files[0])
        print(f"Loading EEG data from: {set_path}")
        raw = mne.io.read_raw_eeglab(set_path, preload=True)

        if 'boundary' in raw.annotations.description:
            print("⚠️ Data contains 'boundary' events. Be cautious with filtering/epoching.")
        
        eog_chs = [ch for ch in raw.info['ch_names'] if ch.startswith('EXG')]
        if eog_chs:
            raw.set_channel_types({ch: 'eog' for ch in eog_chs})

        raw.set_montage('standard_1020', on_missing='ignore')

        X, feature_names = preprocess_and_extract_features(raw, duration=duration, overlap=overlap)

        if X is None:
            print(f"⚠️ No features extracted for {subject} {session}.")
            return [], [], []

        y = [label] * len(X)
        print(f"✅ Extracted {len(X)} samples for {subject} {session}.")
        
        elapsed = time.time() - start_time
        mem = psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2)
        print(f"⏱️ Time taken: {elapsed:.2f} seconds | 🧠 Memory used: {mem:.2f} MB")

        return X, y, feature_names

    except Exception as e:
        print(f"❌ Error processing {subject}, session {session}: {e}")
        return [], [], []

bids_root = "PD3"
subject = "sub-021"
session = "ses-01"
label = 1  # for example

X, y, feat_names = process_subject_session(subject, session, label, bids_root)


## Step 3: Loop Over All Subjects

In [ ]:
import os
import numpy as np
from collections import defaultdict

def categorize_session(session_name, subj_dir):
    """
    Categorize session folders into labels:
    - If subject has both ses-01 and ses-02 folders:
        ses-01 = 'pd-off', ses-02 = 'pd-on'
    - If subject only has ses-01:
        ses-01 = 'ctl'
    - Else:
        'unknown'
    """
    sessions = [d.lower() for d in os.listdir(subj_dir) if os.path.isdir(os.path.join(subj_dir, d))]
    session_name = session_name.lower()
    
    if 'ses-02' in sessions:
        if session_name == 'ses-01':
            return 'pd-off'
        elif session_name == 'ses-02':
            return 'pd-on'
    else:
        if session_name == 'ses-01':
            return 'ctl'
    return 'unknown'


def loop_all_subjects(bids_root, duration=2.0, overlap=1.0):
    label_map = {'ctl': 0, 'pd-off': 1, 'pd-on': 2}
    label_counts = defaultdict(int)
    subject_sample_counts = defaultdict(lambda: defaultdict(int))  # group -> subject -> count
    
    all_X, all_y = [], []

    # List subject folders starting with 'sub-'
    folder_subjects = sorted([d for d in os.listdir(bids_root) if d.startswith('sub-')])
    print(f"\nSubjects found in dataset folder ({bids_root}):")
    print(folder_subjects)

    for subj in folder_subjects:
        subj_dir = os.path.join(bids_root, subj)
        if not os.path.isdir(subj_dir):
            print(f"Subject folder missing or not a dir: {subj_dir}")
            continue

        sessions = [d for d in os.listdir(subj_dir) if os.path.isdir(os.path.join(subj_dir, d))]
        print(f"\nSubject {subj} sessions found: {sessions}")

        for session in sessions:
            label = categorize_session(session, subj_dir)
            print(f"Session {session} categorized as {label}")
            if label == 'unknown':
                print(f"Skipping unknown session {session}")
                continue
            
            numeric_label = label_map[label]

            # process_subject_session should return (X, y, other_info)
            X, y, _ = process_subject_session(subj, session, numeric_label, bids_root, duration, overlap)

            if X is None or len(X) == 0:
                print(f"No data extracted for {subj} {session}")
                continue

            all_X.append(X)
            all_y.extend(y)

            label_counts[label] += len(y)
            subject_sample_counts[label][subj] += len(y)

    if len(all_X) == 0:
        print("No data found or processed.")
        return None, None

    # Stack all features vertically
    X = np.vstack(all_X)
    y = np.array(all_y)

    # Reporting summary
    print("\n📊 Summary:")
    print(f"PD-ON : {len(subject_sample_counts['pd-on'])} subjects, {label_counts['pd-on']} samples")
    print(f"PD-OFF: {len(subject_sample_counts['pd-off'])} subjects, {label_counts['pd-off']} samples")
    print(f"CTL   : {len(subject_sample_counts['ctl'])} subjects, {label_counts['ctl']} samples")

    print(f"\n🧠 Feature matrix shape: X = {X.shape}, y = {y.shape}")
    print(f"🧬 Features per epoch: {X.shape[1] if X.size else 'N/A'}")
    print(f"🎯 Label counts: {dict(zip(*np.unique(y, return_counts=True)))}")

    print("\n⚖️ Class Distribution:")
    total = len(y)
    for group, code in label_map.items():
        pct = (np.sum(y == code) / total) * 100 if total > 0 else 0
        print(f"{group.upper():<6}: {np.sum(y == code):>5} samples ({pct:.1f}%)")

    # Optional: comment out if too verbose
    print("\n👥 Subject Sample Breakdown:")
    for group in label_map.keys():
        print(f"{group.upper():<6}:")
        for subj, count in subject_sample_counts[group].items():
            print(f"   {subj}: {count} samples")

    return X, y


# Example usage (make sure process_subject_session is defined):
X, y = loop_all_subjects(bids_root='PD3')


### Debug

## Step 4: Train ML Classifier

In [ ]:
import os
import numpy as np
from collections import defaultdict

def categorize_session(session_name, subj_dir):
    """
    Categorize session folders into labels:
    - If subject has both ses-01 and ses-02 folders:
        ses-01 = 'pd-off', ses-02 = 'pd-on'
    - If subject only has ses-01:
        ses-01 = 'ctl'
    - Else:
        'unknown'
    """
    sessions = [d.lower() for d in os.listdir(subj_dir) if os.path.isdir(os.path.join(subj_dir, d))]
    session_name = session_name.lower()
    
    if 'ses-02' in sessions:
        if session_name == 'ses-01':
            return 'pd-off'
        elif session_name == 'ses-02':
            return 'pd-on'
    else:
        if session_name == 'ses-01':
            return 'ctl'
    return 'unknown'


def loop_all_subjects(bids_root, duration=2.0, overlap=1.0):
    label_map = {'ctl': 0, 'pd-off': 1, 'pd-on': 2}
    label_counts = defaultdict(int)
    subject_sample_counts = defaultdict(lambda: defaultdict(int))  # group -> subject -> count
    
    all_X, all_y, all_subjects = [], [], []  # 👈 added subjects list

    # List subject folders starting with 'sub-'
    folder_subjects = sorted([d for d in os.listdir(bids_root) if d.startswith('sub-')])
    print(f"\nSubjects found in dataset folder ({bids_root}):")
    print(folder_subjects)

    for subj in folder_subjects:
        subj_dir = os.path.join(bids_root, subj)
        if not os.path.isdir(subj_dir):
            print(f"Subject folder missing or not a dir: {subj_dir}")
            continue

        sessions = [d for d in os.listdir(subj_dir) if os.path.isdir(os.path.join(subj_dir, d))]
        print(f"\nSubject {subj} sessions found: {sessions}")

        for session in sessions:
            label = categorize_session(session, subj_dir)
            print(f"Session {session} categorized as {label}")
            if label == 'unknown':
                print(f"Skipping unknown session {session}")
                continue
            
            numeric_label = label_map[label]

            # process_subject_session should return (X, y, other_info)
            X, y, _ = process_subject_session(subj, session, numeric_label, bids_root, duration, overlap)

            if X is None or len(X) == 0:
                print(f"No data extracted for {subj} {session}")
                continue

            all_X.append(X)
            all_y.extend(y)

            # 👇 repeat subject ID for each epoch
            all_subjects.extend([subj] * len(y))

            label_counts[label] += len(y)
            subject_sample_counts[label][subj] += len(y)

    if len(all_X) == 0:
        print("No data found or processed.")
        return None, None, None

    # Stack all features vertically
    X = np.vstack(all_X)
    y = np.array(all_y)
    subjects = np.array(all_subjects)   # 👈 per-epoch subject IDs

    # Reporting summary
    print("\n📊 Summary:")
    print(f"PD-ON : {len(subject_sample_counts['pd-on'])} subjects, {label_counts['pd-on']} samples")
    print(f"PD-OFF: {len(subject_sample_counts['pd-off'])} subjects, {label_counts['pd-off']} samples")
    print(f"CTL   : {len(subject_sample_counts['ctl'])} subjects, {label_counts['ctl']} samples")

    print(f"\n🧠 Feature matrix shape: X = {X.shape}, y = {y.shape}, subjects = {subjects.shape}")
    print(f"🧬 Features per epoch: {X.shape[1] if X.size else 'N/A'}")
    print(f"🎯 Label counts: {dict(zip(*np.unique(y, return_counts=True)))}")

    print("\n⚖️ Class Distribution:")
    total = len(y)
    for group, code in label_map.items():
        pct = (np.sum(y == code) / total) * 100 if total > 0 else 0
        print(f"{group.upper():<6}: {np.sum(y == code):>5} samples ({pct:.1f}%)")

    print("\n👥 Subject Sample Breakdown:")
    for group in label_map.keys():
        print(f"{group.upper():<6}:")
        for subj, count in subject_sample_counts[group].items():
            print(f"   {subj}: {count} samples")

    return X, y, subjects


### Train Random Forest Classifier

In [ ]:
def subject_wise_split(X, y, subjects, test_size=0.2, random_state=42):
    unique_subjects = np.unique(subjects)
    subj_train, subj_test = train_test_split(
        unique_subjects, test_size=test_size, random_state=random_state
    )

    train_mask = np.isin(subjects, subj_train)
    test_mask = np.isin(subjects, subj_test)

    X_train, y_train = X[train_mask], y[train_mask]
    X_test, y_test = X[test_mask], y[test_mask]
    subj_train_epochs, subj_test_epochs = subjects[train_mask], subjects[test_mask]

    print(f"Train: {len(subj_train)} subjects, {len(X_train)} epochs")
    print(f"Test : {len(subj_test)} subjects, {len(X_test)} epochs")

    return X_train, X_test, y_train, y_test, subj_train_epochs, subj_test_epochs


In [ ]:
# Step 1: build full dataset
X, y, subjects = loop_all_subjects(bids_root="PD3", duration=2.0, overlap=1.0)

# Step 2: split by subject (not by epochs)
X_train, X_test, y_train, y_test, subj_train, subj_test = subject_wise_split(
    X, y, subjects, test_size=0.2
)

In [ ]:
from sklearn.model_selection import train_test_split

# Assuming X and y have been processed and are available

# Split data into training and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Print the shapes of the resulting datasets
print(f"✅ Shape of X_train: {X_train.shape}")
print(f"✅ Shape of X_test: {X_test.shape}")
print(f"✅ Shape of y_train: {y_train.shape}")
print(f"✅ Shape of y_test: {y_test.shape}")

# Check the first epoch in the dataset
print(f"First sample (epoch) shape: {X[0].shape}")
print(f"First sample (epoch) data: {X[0]}")



In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------
# MODEL NAME & VARIABLES
# ------------------------
model_name = "Random Forest"
class_names = ['Control', 'PD-On', 'PD-Off']
n_classes = len(class_names)

# Ensure y is a NumPy array
if isinstance(y, pd.Series):
    y = y.values
elif isinstance(y, pd.DataFrame):
    y = y.squeeze().values

# Storage for all folds
fold_results = []
all_y_prob_rf = []  # For probabilities across all folds
all_y_true_rf = []  # For true labels across all folds

# ------------------------
# CROSS-VALIDATION LOOP
# ------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n📁 Fold {fold}")

    # Split
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Standardize
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Model
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=20
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train_scaled, y_train)

    # Predictions
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)

    # Save for aggregate ROC later
    all_y_prob_rf.append(y_proba)
    all_y_true_rf.append(y_test)

    # Metrics
    acc = np.mean(y_pred == y_test)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')
    y_test_bin = label_binarize(y_test, classes=list(range(n_classes)))
    auc_score = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')

    fold_results.append({
        "fold": fold,
        "model": model_name,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "auc": auc_score
    })

    # Reports
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"AUC Score: {auc_score:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=class_names))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - Fold {fold}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

# ------------------------
# AGGREGATE STORAGE
# ------------------------
all_y_prob_rf = np.vstack(all_y_prob_rf)  # Shape: (total_samples, n_classes)
all_y_true_rf = np.concatenate(all_y_true_rf)  # Shape: (total_samples,)

# ------------------------
# SUMMARY
# ------------------------
results_df = pd.DataFrame(fold_results)
print("\n📊 Summary Across Folds:")
print(results_df.groupby("model").mean(numeric_only=True))

# Example boxplot
plt.figure(figsize=(7, 5))
sns.boxplot(data=results_df, x="model", y="accuracy")
plt.title(f"{model_name} Accuracy Across Folds")
plt.ylabel("Accuracy")
plt.xlabel("Model")
plt.tight_layout()
plt.show()


# ***Decision Tree***

In [ ]:
!pip install plotly


In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------
# MODEL NAME & VARIABLES
# ------------------------
model_name = "Decision Tree"
class_names = ['Control', 'PD-On', 'PD-Off']
n_classes = len(class_names)

# Ensure y is a NumPy array
if isinstance(y, pd.Series):
    y = y.values
elif isinstance(y, pd.DataFrame):
    y = y.squeeze().values

# Storage for results
fold_results = []
all_y_prob_dt = []  # Probabilities across folds
all_y_true_dt = []  # True labels across folds

# ------------------------
# CROSS-VALIDATION LOOP
# ------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n📁 Fold {fold}")

    # Split
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Standardize
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Model
    model = DecisionTreeClassifier(
        max_depth=20,
        class_weight='balanced',
        random_state=42
    )
    model.fit(X_train_scaled, y_train)

    # Predictions
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)

    # Save for later aggregate analysis
    all_y_prob_dt.append(y_proba)
    all_y_true_dt.append(y_test)

    # Metrics
    acc = np.mean(y_pred == y_test)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')
    y_test_bin = label_binarize(y_test, classes=list(range(n_classes)))
    auc_score = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')

    fold_results.append({
        "fold": fold,
        "model": model_name,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "auc": auc_score
    })

    # Reports
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"AUC Score: {auc_score:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=class_names))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - Fold {fold}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

# ------------------------
# AGGREGATE STORAGE
# ------------------------
all_y_prob_dt = np.vstack(all_y_prob_dt)  # Shape: (total_samples, n_classes)
all_y_true_dt = np.concatenate(all_y_true_dt)  # Shape: (total_samples,)

# ------------------------
# SUMMARY
# ------------------------
results_df = pd.DataFrame(fold_results)
print("\n📊 Summary Across Folds:")
print(results_df.groupby("model").mean(numeric_only=True))

# Example accuracy plot
plt.figure(figsize=(7, 5))
sns.boxplot(data=results_df, x="model", y="accuracy")
plt.title(f"{model_name} Accuracy Across Folds")
plt.ylabel("Accuracy")
plt.xlabel("Model")
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

# === Plot the Final Decision Tree Model with Adjustments ===
plt.figure(figsize=(150, 75))  # Decrease the size of the figure to avoid overcrowding

# Visualizing the decision tree using plot_tree function
plot_tree(
    model,  # The trained decision tree model
    filled=True,  # Color the nodes to make them more readable
    feature_names=['Feature' + str(i) for i in range(X_train_scaled.shape[1])],  # Feature names (e.g., Feature0, Feature1, ...)
    class_names=class_names,  # Class names for each target class (Control, PD-On, PD-Off)
    rounded=True,  # Round the corners of the nodes for better aesthetics
    fontsize=10,  # Reduce font size to avoid overcrowding
    proportion=True,  # Show proportion of samples in each node
    precision=2  # Precision of float values displayed on the tree
)

# Add a title to the plot
plt.title("Final Decision Tree Visualization", fontsize=8)

# Adjust layout to avoid clipping of labels and titles
plt.subplots_adjust(left=0.05, right=0.95, top=0.9, bottom=0.05)

# Display the plot
plt.show()



### Step 4.2 : Train Support Vector Machine Classifier

In [ ]:
# import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------------
# MODEL NAME & VARIABLES
# ------------------------
model_name = "SVM"
class_names = ['Control', 'PD-On', 'PD-Off']
n_classes = len(class_names)

# Ensure y is a NumPy array
if isinstance(y, pd.Series):
    y = y.values
elif isinstance(y, pd.DataFrame):
    y = y.squeeze().values

# Storage for results
fold_results = []
all_y_prob_svm = []  # Probabilities across folds
all_y_true_svm = []  # True labels across folds

# ------------------------
# CROSS-VALIDATION LOOP
# ------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n📁 Fold {fold}")

    # Split
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Standardize
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Model
    model = SVC(
        kernel='linear',         # Adjustable kernel
        C=1,                     # Regularization parameter
        class_weight='balanced', # Handle class imbalance
        probability=True,        # Enable probability estimates
        random_state=42
    )
    model.fit(X_train_scaled, y_train)

    # Predictions
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)

    # Save for later aggregate analysis
    all_y_prob_svm.append(y_proba)
    all_y_true_svm.append(y_test)

    # Metrics
    acc = np.mean(y_pred == y_test)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')
    y_test_bin = label_binarize(y_test, classes=list(range(n_classes)))
    auc_score = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')

    fold_results.append({
        "fold": fold,
        "model": model_name,
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "auc": auc_score
    })

    # Reports
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"AUC Score: {auc_score:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=class_names))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - Fold {fold}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

# ------------------------
# AGGREGATE STORAGE
# ------------------------
all_y_prob_svm = np.vstack(all_y_prob_svm)  # Shape: (total_samples, n_classes)
all_y_true_svm = np.concatenate(all_y_true_svm)  # Shape: (total_samples,)

# ------------------------
# SUMMARY
# ------------------------
results_df = pd.DataFrame(fold_results)
print("\n📊 Summary Across Folds:")
print(results_df.groupby("model").mean(numeric_only=True))

# Example accuracy plot
plt.figure(figsize=(7, 5))
sns.boxplot(data=results_df, x="model", y="accuracy")
plt.title(f"{model_name} Accuracy Across Folds")
plt.ylabel("Accuracy")
plt.xlabel("Model")
plt.tight_layout()
plt.show()


## Step 4.3: Train Logistic Regression Classifier

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_fscore_support

# Assuming X and y have been processed and are available
if isinstance(y, pd.Series):
    y = y.values
elif isinstance(y, pd.DataFrame):
    y = y.squeeze().values

# 🎯 Class names
class_names = ['Control', 'PD-On', 'PD-Off']
n_classes = len(class_names)

# ⚙️ Cross-Validation Setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 🧪 Metrics tracking
all_metrics = {
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": [],
    "auc": []
}

# 📥 Store all predictions for ROC curves later
all_y_true_logreg = []
all_y_prob_logreg = []

# 🔄 Fold Loop
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n📁 Fold {fold}")
    
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # ⚖️ Preprocessing
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 🌳 Logistic Regression Model
    model = LogisticRegression(
        max_iter=1000,
        class_weight='balanced',
        random_state=42,
        multi_class='ovr',
        solver='liblinear'
    )
    model.fit(X_train_scaled, y_train)
    
    # Predictions
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)
    
    # Append for overall storage
    all_y_true_logreg.extend(y_test)
    all_y_prob_logreg.extend(y_proba)
    
    # Metrics
    acc = np.mean(y_pred == y_test)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')
    y_test_bin = label_binarize(y_test, classes=list(range(n_classes)))
    auc_score = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')

    all_metrics["accuracy"].append(acc)
    all_metrics["precision"].append(prec)
    all_metrics["recall"].append(rec)
    all_metrics["f1"].append(f1)
    all_metrics["auc"].append(auc_score)

    # Reports
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"AUC Score: {auc_score:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=class_names))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - Fold {fold}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

    # ROC Curve
    fpr, tpr = {}, {}
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    
    plt.figure(figsize=(6, 5))
    plt.plot(all_fpr, mean_tpr, color='blue', label=f'Macro-Averaged ROC (AUC = {auc_score:.2f})')
    plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Chance')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Macro ROC Curve - Fold {fold}')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

# Convert to numpy arrays for compatibility
all_y_true_logreg = np.array(all_y_true_logreg)
all_y_prob_logreg = np.array(all_y_prob_logreg)

# 📊 Overall Summary
print("\n📊 Overall Cross-Validation Summary:")
for key in all_metrics:
    mean_val = np.mean(all_metrics[key])
    print(f"Mean {key.capitalize():<9}: {mean_val:.4f}")


## **Transformer Model**

In [ ]:
!pip install torch torchvision

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

**BLOCK 1: Preprocessing and DataLoader**

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import torch
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# Entire dataset (X and y)
X = np.concatenate([X_train, X_test], axis=0)  # shape (4497, 160)
y = np.concatenate([y_train, y_test], axis=0)  # shape (4497,)

X = X.astype(np.float32)
y = y.astype(np.int64)

# K-Fold cross-validation will split this
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Store all folds' loaders, weights, scalers
fold_data = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"📂 Preparing Fold {fold_idx}...")

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # Standardize using only training set
    scaler = StandardScaler()
    X_train_std = scaler.fit_transform(X_train)
    X_val_std = scaler.transform(X_val)

    # Convert to PyTorch tensors
    X_train_tensor = torch.tensor(X_train_std, dtype=torch.float32)
    X_val_tensor = torch.tensor(X_val_std, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long)
    y_val_tensor = torch.tensor(y_val, dtype=torch.long)

    # Compute class weights for this fold
    class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

    # Create DataLoaders
    batch_size = 64
    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=batch_size)

    fold_data.append({
        'train_loader': train_loader,
        'val_loader': val_loader,
        'class_weights': class_weights_tensor,
        'scaler': scaler,
        'X_val': X_val_std,
        'y_val': y_val,
    })



**BLOCK 2: Transformer Model Definition**

In [ ]:
import torch
import torch.nn as nn

class EEGTransformer(nn.Module):
    def __init__(self, input_dim=160, num_classes=3, embed_dim=128, num_heads=4, num_layers=2, dropout=0.3):
        super().__init__()
        
        self.embedding = nn.Linear(input_dim, embed_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=num_heads, 
            dropout=dropout,
            batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        # x shape: (B, input_dim)
        x = self.embedding(x).unsqueeze(1)  # (B, 1, embed_dim)
        x = self.transformer_encoder(x)     # (B, 1, embed_dim)
        x = x.squeeze(1)                    # (B, embed_dim)
        return self.classifier(x)           # (B, num_classes)


**BLOCK 3: Training Setup**

In [ ]:
from transformers import get_cosine_schedule_with_warmup
import torch.optim as optim
import torch.nn as nn

def initialize_training_components(input_dim, num_classes, class_weights_tensor, train_loader, epochs=1000, lr=5e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Initialize model
    model = EEGTransformer(input_dim=input_dim, num_classes=num_classes).to(device)

    # Loss function with class weights
    loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))

    # Optimizer
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    # Cosine LR scheduler with warmup
    total_steps = len(train_loader) * epochs
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),  # 10% warmup by default
        num_training_steps=total_steps
    )

    return model, loss_fn, optimizer, scheduler, device


**BLOCK 4: Training and Evaluation Loop with tqdm**

In [ ]:
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import torch
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

def train_one_epoch(model, loader, optimizer, loss_fn, scheduler, device):
    model.train()
    total_loss = 0
    correct, total = 0, 0
    loop = tqdm(loader, desc="Train", leave=False)
    
    for xb, yb in loop:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        correct += (preds.argmax(1) == yb).sum().item()
        total += yb.size(0)

        loop.set_postfix(loss=loss.item(), acc=correct / total)

    avg_loss = total_loss / len(loader)
    avg_acc = correct / total
    return avg_loss, avg_acc

def evaluate(model, loader, device, return_preds=False, show_plot=True):
    model.eval()
    y_true, y_pred, y_prob = [], [], []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            preds = logits.argmax(1).cpu().numpy()

            y_prob.extend(probs)
            y_pred.extend(preds)
            y_true.extend(yb.numpy())

    if return_preds:
        return y_true, y_pred, y_prob

    # Report and Confusion Matrix
    print("\n📊 Classification Report:")
    print(classification_report(y_true, y_pred, target_names=["CTL", "PD-On", "PD-Off"], digits=4))

    cm = confusion_matrix(y_true, y_pred)
    df_cm = pd.DataFrame(cm, index=["CTL", "PD-On", "PD-Off"], columns=["CTL", "PD-On", "PD-Off"])

    plt.figure(figsize=(6, 5))
    sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix")
    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.show()


**BLOCK 5: Training Loop**

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def plot_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    df_cm = pd.DataFrame(cm, index=["CTL", "PD-On", "PD-Off"], columns=["CTL", "PD-On", "PD-Off"])
    plt.figure(figsize=(6, 5))
    sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix")
    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.show()

def plot_macro_roc_curve(y_true, y_prob):
    y_true_bin = label_binarize(y_true, classes=[0, 1, 2])
    y_prob = np.array(y_prob)
    fpr, tpr, _ = roc_curve(y_true_bin.ravel(), y_prob.ravel())
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f"Macro-average ROC curve (AUC = {roc_auc:.4f})")
    plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Macro-Averaged ROC Curve")
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# Updated training + evaluation loop for Transformer model
num_epochs = 40
all_y_true_transformer, all_y_pred_transformer, all_y_prob_transformer = [], [], []

for fold_idx, fold in enumerate(fold_data, 1):
    print(f"\n📁 ===== Fold {fold_idx} =====")

    train_loader = fold["train_loader"]
    val_loader = fold["val_loader"]
    class_weights = fold["class_weights"]

    model, loss_fn, optimizer, scheduler, device = initialize_training_components(
        input_dim=160,
        num_classes=3,
        class_weights_tensor=class_weights,
        train_loader=train_loader,
        epochs=num_epochs,
        lr=5e-4
    )

    # Train loop
    for epoch in range(1, num_epochs + 1):
        print(f"\n🌟 Epoch {epoch}/{num_epochs} — Fold {fold_idx}")
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, loss_fn, scheduler, device)
        print(f"✅ Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")

    # Final evaluation on validation set for this fold
    y_true, y_pred, y_prob = evaluate(model, val_loader, device, return_preds=True, show_plot=False)

    # Print classification report
    print("\n📊 Classification Report:")
    print(classification_report(y_true, y_pred, target_names=["CTL", "PD-On", "PD-Off"], digits=4))

    # Plot confusion matrix
    plot_confusion_matrix(y_true, y_pred)

    # Plot macro-average ROC AUC curve
    plot_macro_roc_curve(y_true, y_prob)

    # Collect all fold data for combined stats later
    all_y_true_transformer.extend(y_true)
    all_y_pred_transformer.extend(y_pred)
    all_y_prob_transformer.extend(y_prob)

**BLOCK 6:Evauluate**

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

# --- Final Classification Report and Confusion Matrix ---
print("\n📊 Final Cross-Validated Classification Report (Transformer):")
print(classification_report(
    all_y_true_transformer, all_y_pred_transformer,
    target_names=["CTL", "PD-On", "PD-Off"],
    digits=4
))

cm = confusion_matrix(all_y_true_transformer, all_y_pred_transformer)
df_cm = pd.DataFrame(cm, index=["CTL", "PD-On", "PD-Off"], columns=["CTL", "PD-On", "PD-Off"])

plt.figure(figsize=(6, 5))
sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Confusion Matrix (Transformer - All Folds Combined)", fontsize=13)
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.show()


# --- Macro-Averaged ROC Curve ---

# Binarize labels for multiclass ROC
y_true_bin = label_binarize(all_y_true_transformer, classes=[0, 1, 2])
y_prob = np.array(all_y_prob_transformer)

n_classes = y_true_bin.shape[1]
fpr = dict()
tpr = dict()
roc_auc = dict()

# Compute ROC curve and AUC for each class
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Aggregate all FPR points
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))

# Interpolate all TPRs at these points and average them
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])

mean_tpr /= n_classes

# Compute macro-average AUC
macro_auc = auc(all_fpr, mean_tpr)

# Plot macro-average ROC
plt.figure(figsize=(7, 6))
plt.plot(all_fpr, mean_tpr, color='navy', lw=2, label=f'Macro-average ROC (AUC = {macro_auc:.4f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('📈 Macro-Averaged ROC Curve (Transformer - All Folds Combined)', fontsize=13)
plt.legend(loc='lower right')
plt.grid(True, linestyle=':')
plt.tight_layout()
plt.show()



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

# Convert all y_prob lists to numpy arrays once to avoid errors
all_y_prob_svm = np.array(all_y_prob_svm)
all_y_prob_transformer = np.array(all_y_prob_transformer)
all_y_prob_logreg = np.array(all_y_prob_logreg)
all_y_prob_dt = np.array(all_y_prob_dt)
all_y_prob_rf = np.array(all_y_prob_rf)

# Dictionary with your models' true labels and predicted probabilities
model_data = {
    "SVM": (all_y_true_svm, all_y_prob_svm),
    "Transformer": (all_y_true_transformer, all_y_prob_transformer),
    "Logistic Regression": (all_y_true_logreg, all_y_prob_logreg),
    "Decision Tree": (all_y_true_dt, all_y_prob_dt),
    "Random Forest": (all_y_true_rf, all_y_prob_rf)
}

class_names = ['Control', 'PD-On', 'PD-Off']
n_classes = len(class_names)

plt.figure(figsize=(10, 8))

for model_name, (y_true, y_proba) in model_data.items():
    # Binarize true labels for multiclass ROC calculation
    y_true_bin = label_binarize(y_true, classes=list(range(n_classes)))

    fpr = dict()
    tpr = dict()

    # Calculate ROC curve and AUC for each class
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_proba[:, i])

    # Combine all false positive rates for interpolation
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))

    # Interpolate true positive rates and calculate mean TPR (macro-average)
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes

    # Compute macro-average AUC
    macro_auc = auc(all_fpr, mean_tpr)

    # Plot the macro-average ROC curve for this model
    plt.plot(all_fpr, mean_tpr, lw=2, label=f'{model_name} (AUC = {macro_auc:.3f})')

# Plot random chance line
plt.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='Chance')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('📈 Macro-Averaged ROC Curves - All Models')
plt.legend(loc='lower right')
plt.grid(True, linestyle=':')
plt.tight_layout()
plt.show()
